# 01 – Data preparation

This notebook walks through the data preparation part of the project, step by
step:

1. [The IMDb tables](#1.-The-IMDb-tables): what each table contains and how
   the tables are linked.
2. [Missing values](#2.-Missing-values): share of null / empty values per
   column, the five emptiest columns and why they are empty.
3. [Other data quality issues](#3.-Other-data-quality-issues): impossible or
   suspicious values, broken links between tables, duplicates.
4. [Handling the issues](#4.-Handling-the-issues): the cleaning rules and
   their effect on the data.
5. [Using the clean tables](#5.-Using-the-clean-tables).

**Prerequisites.** The heavy computations are done by the jobs (they scan
more than 200 million rows and take several minutes each). Run them first:

```bash
docker compose exec spark python jobs/ingest_imdb.py      # raw  -> bronze
docker compose exec spark python jobs/missing_values.py   # report 2
docker compose exec spark python jobs/quality_checks.py   # report 3
docker compose exec spark python jobs/prepare_data.py     # bronze -> silver
```

This notebook reads their reports (small CSV files in `data/reports/`) and
runs smaller queries on the tables to explain the results. The code of every
rule is in `src/imdb_project/quality.py` and
`src/imdb_project/preparation.py`.

In [1]:
import pandas as pd
from pyspark.sql import functions as F

from imdb_project.config import IMDB_TABLES, REPORTS_DIR
from imdb_project.io import read_bronze, read_silver
from imdb_project.quality import ARCHIVE_CATEGORIES, FIRST_FILM_YEAR, MAX_PLAUSIBLE_AGE
from imdb_project.spark import get_spark

spark = get_spark("notebook-data-preparation")
spark.sparkContext.setLogLevel("ERROR")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)


def report(name):
    """Read a CSV report written by the jobs into pandas."""
    files = sorted((REPORTS_DIR / name).glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"Run the job that writes data/reports/{name} first")
    return pd.read_csv(files[0])


bronze = {t: read_bronze(spark, t) for t in IMDB_TABLES}
silver = {t: read_silver(spark, t) for t in IMDB_TABLES}

## 1. The IMDb tables

The dataset is built around **two central tables**: titles (`title.basics`,
key `tconst`, ids such as `tt0111161`) and people (`name.basics`, key
`nconst`, ids such as `nm0000151`). The other tables point to them:

```
                 title.ratings (rating, votes)
                        | tconst
title.akas --titleId-- title.basics --tconst-- title.episode (episode -> parent series)
(other titles)          | tconst
                        |-- title.crew        (lists of directors / writers)
                        `-- title.principals  (main credits)
                                 | nconst
                           name.basics (people)
```

| Table | One row is | Content |
|---|---|---|
| `title.basics` | a title | type (`movie`, `short`, `tvSeries`, `tvEpisode`, `videoGame`...), titles, start / end year, runtime, up to 3 genres |
| `name.basics` | a person | name, birth / death year, up to 3 professions, up to 4 "known for" titles |
| `title.principals` | a credit (title, person) | the main link between titles and people: credit order, category (`actor`, `director`, `writer`, `composer`, `archive_footage`...), job, characters |
| `title.crew` | a title | all the directors and writers, as lists of `nconst` |
| `title.akas` | an alternative title | the title used in a region and / or language |
| `title.episode` | an episode | parent series, season and episode numbers |
| `title.ratings` | a rated title | average rating and number of votes |

The ingestion job (`jobs/ingest_imdb.py`) copies the TSV files to Parquet
without changing them: every column is read as text and `\N`, the IMDb code
for "no value", is read as null. This copy is the **bronze** layer.

In [2]:
counts = pd.DataFrame(
    [(t, bronze[t].count(), len(bronze[t].columns)) for t in IMDB_TABLES],
    columns=["table", "rows", "columns"],
)
counts

,table,rows,columns
0,name.basics,15802420,6
1,title.akas,60321594,8
2,title.basics,12951113,9
3,title.crew,12949786,3
4,title.episode,10025088,4
5,title.principals,102649609,6
6,title.ratings,1730885,3


Most titles are **TV episodes**; feature films (`movie`) are a small part of
the table. This matters for the rest of the project: every question about
films starts by selecting the right title type.

In [3]:
(bronze["title.basics"].groupBy("titleType").count()
 .orderBy(F.desc("count")).toPandas())

,titleType,count
0,tvEpisode,10025088
1,short,1168455
2,movie,760460
3,video,333318
4,tvSeries,308122
5,tvMovie,156973
6,tvMiniSeries,74154
7,tvSpecial,62363
8,videoGame,51059
9,tvShort,11120


In [4]:
bronze["title.principals"].where("tconst = 'tt0111161'").orderBy(
    F.col("ordering").cast("int")).show(truncate=False)

+---------+--------+---------+-------------------+------------------------------------------------------------------------+----------------------------+
|tconst   |ordering|nconst   |category           |job                                                                     |characters                  |
+---------+--------+---------+-------------------+------------------------------------------------------------------------+----------------------------+
|tt0111161|1       |nm0000209|actor              |NULL                                                                    |["Andy Dufresne"]           |
|tt0111161|2       |nm0000151|actor              |NULL                                                                    |["Ellis Boyd 'Red' Redding"]|
|tt0111161|3       |nm0348409|actor              |NULL                                                                    |["Warden Norton"]           |
|tt0111161|4       |nm0006669|actor              |NULL                            

## 2. Missing values

**Method.** For each column of each table, `jobs/missing_values.py` counts:

- `null_count`: values equal to `\N` in the source (null after ingestion);
- `empty_count`: empty or blank strings;
- `missing_pct`: both together, as a percentage of the rows.

The count is made on the bronze tables, so it describes the source data
before any cleaning. A value is counted as missing when the field is empty,
whatever the reason (see 2.2: many of these values are empty on purpose).

In [5]:
missing = report("missing_values")
missing.sort_values(["table", "missing_pct"], ascending=[True, False])

,table,column,rows,null_count,empty_count,missing_count,null_pct,missing_pct
3,name.basics,deathYear,15802420,15537057,0,15537057,98.3207,98.3207
2,name.basics,birthYear,15802420,15116831,0,15116831,95.6615,95.6615
4,name.basics,primaryProfession,15802420,3216642,0,3216642,20.3554,20.3554
5,name.basics,knownForTitles,15802420,1915685,0,1915685,12.1227,12.1227
1,name.basics,primaryName,15802420,89,0,89,0.0006,0.0006
0,name.basics,nconst,15802420,0,0,0,0.0000,0.0000
12,title.akas,attributes,60321594,60004431,0,60004431,99.4742,99.4742
11,title.akas,types,60321594,40342236,0,40342236,66.8786,66.8786
10,title.akas,language,60321594,19639034,0,19639034,32.5572,32.5572
9,title.akas,region,60321594,12991240,0,12991240,21.5366,21.5366


There is **no empty string** in the whole dataset: every missing value is
coded `\N`, so `missing_pct` is equal to `null_pct`.

In [6]:
print("Empty strings in the dataset:", missing["empty_count"].sum())

Empty strings in the dataset: 0


### 2.1 The five emptiest columns

In [7]:
top5 = missing.sort_values("missing_pct", ascending=False).head(5)
top5[["table", "column", "rows", "missing_count", "missing_pct"]]

,table,column,rows,missing_count,missing_pct
12,title.akas,attributes,60321594,60004431,99.4742
20,title.basics,endYear,12951113,12786793,98.7312
3,name.basics,deathYear,15802420,15537057,98.3207
2,name.basics,birthYear,15802420,15116831,95.6615
34,title.principals,job,102649609,82794746,80.6576


### 2.2 Why are these columns so empty?

A high share of nulls is not always a data quality problem. For each of the
five columns, the queries below look at *where* the nulls are.

#### `title.basics.endYear`

`endYear` is the year a TV series ended. It only makes sense for
`tvSeries` and `tvMiniSeries`, which are about 3 % of the titles. For every
other type (films, episodes, shorts...) it is always null, as it should be.
Among series, a null `endYear` means that the series is still running or
that its end was not entered.

In [8]:
(silver["title.basics"].groupBy("titleType")
 .agg(F.count(F.lit(1)).alias("titles"),
      F.round(F.avg(F.col("endYear").isNull().cast("int")) * 100, 2).alias("endYear_null_pct"))
 .orderBy(F.desc("titles")).toPandas())

,titleType,titles,endYear_null_pct
0,tvEpisode,10025088,100.00
1,short,1168455,100.00
2,movie,760460,100.00
3,video,333318,100.00
4,tvSeries,308122,60.62
5,tvMovie,156973,100.00
6,tvMiniSeries,74154,42.06
7,tvSpecial,62363,100.00
8,videoGame,51059,100.00
9,tvShort,11120,100.00


#### `name.basics.birthYear` and `name.basics.deathYear`

`name.basics` lists everybody who appears in a credit, including millions of
people with one or two minor credits. IMDb rarely has biographical data for
them. The share of known birth years grows with the number of credits a
person has in `title.principals`:

In [9]:
credits = silver["title.principals"].groupBy("nconst").agg(
    F.count(F.lit(1)).alias("credits"))
people = silver["name.basics"].join(credits, "nconst", "left").fillna(0, ["credits"])
bucket = (F.when(F.col("credits") == 0, "0")
          .when(F.col("credits") <= 2, "1-2")
          .when(F.col("credits") <= 10, "3-10")
          .when(F.col("credits") <= 100, "11-100")
          .otherwise("over 100"))
(people.withColumn("principal_credits", bucket)
 .groupBy("principal_credits")
 .agg(F.count(F.lit(1)).alias("people"),
      F.round(F.avg(F.col("birthYear").isNull().cast("int")) * 100, 1).alias("birthYear_null_pct"))
 .orderBy("people").toPandas())

,principal_credits,people,birthYear_null_pct
0,over 100,163799,50.0
1,11-100,799383,73.4
2,3-10,1459951,89.4
3,1-2,4839111,97.4
4,0,8540176,98.7


`deathYear` is null for two reasons: the person is alive, or nothing is known
about the person at all. When the birth year is known, the death year is
filled for about a third of the people; the others are mostly alive. Some of
them are not: people born more than `MAX_PLAUSIBLE_AGE` years ago without a
death year are flagged in part 3 (`probably_missing_death_year`).

In [10]:
names = silver["name.basics"]
names.select(
    F.round(F.avg(F.col("deathYear").isNull().cast("int")) * 100, 2).alias("deathYear_null_pct_all"),
    F.round(F.avg(F.when(F.col("birthYear").isNotNull(),
                         F.col("deathYear").isNull().cast("int"))) * 100, 2)
     .alias("deathYear_null_pct_when_birth_known"),
).toPandas()

,deathYear_null_pct_all,deathYear_null_pct_when_birth_known
0,98.32,63.8


#### `title.principals.job`

`job` gives details on the role when the `category` is not enough. It is
never filled for actors, actresses, people playing themselves (`self`) or
archive footage, which are more than half of the credits. For producers it
is always filled (`producer`, `executive producer`...).

In [11]:
(silver["title.principals"].groupBy("category")
 .agg(F.count(F.lit(1)).alias("credits"),
      F.round(F.avg(F.col("job").isNull().cast("int")) * 100, 1).alias("job_null_pct"))
 .orderBy(F.desc("credits")).toPandas())

,category,credits,job_null_pct
0,actor,24288690,100.0
1,actress,18349320,100.0
2,self,15585869,100.0
3,writer,12205509,39.9
4,director,8672889,83.0
5,producer,7565859,0.0
6,editor,5565855,78.1
7,cinematographer,4115782,73.2
8,composer,3256344,78.2
9,production_designer,1196405,82.9


#### `title.akas.attributes`

`attributes` describes an alternative title in free words ("literal English
title", "transliterated title"...). Most alternative titles are just the
title used in a country and need no description, so the column is almost
always empty.

In [12]:
(silver["title.akas"].select(F.explode("attributes").alias("attribute"))
 .groupBy("attribute").count().orderBy(F.desc("count")).limit(10).toPandas())

,attribute,count
0,transliterated title,31712
1,alternative spelling,24790
2,literal English title,21177
3,complete title,19947
4,literal title,16996
5,new title,16433
6,video box title,13067
7,series title,12907
8,short title,12887
9,alternative transliteration,12669


**Conclusion.** The five emptiest columns are empty *by design* (the field
does not apply: `endYear`, `job`, `attributes`) or because IMDb has little
information on minor credits (`birthYear`, `deathYear`). They are kept as
they are: no value is invented (imputed). In particular `runtimeMinutes`,
missing for most titles, is not filled with an average, because the
exploration part studies runtimes and invented values would bias it.

### 2.3 Values that cannot be converted

Before casting the text columns to numbers, `jobs/quality_checks.py` counts
the values that are not null but cannot be converted (for instance a year
containing letters). There are none.

In [13]:
casts = report("invalid_casts")
print("Values that cannot be converted:", casts["invalid_values"].sum())
casts.head(10)

Values that cannot be converted: 0


,table,column,type,invalid_values
0,name.basics,birthYear,int,0
1,name.basics,deathYear,int,0
2,title.akas,ordering,int,0
3,title.akas,isOriginalTitle,int,0
4,title.basics,isAdult,int,0
5,title.basics,startYear,int,0
6,title.basics,endYear,int,0
7,title.basics,runtimeMinutes,int,0
8,title.episode,seasonNumber,int,0
9,title.episode,episodeNumber,int,0


## 3. Other data quality issues

`jobs/quality_checks.py` runs a list of checks, grouped in six categories:

| Category | Meaning | Example |
|---|---|---|
| `impossible_value` | the value cannot be true | death before birth, runtime of 0 minutes |
| `implausible_value` | possible but suspicious | runtime over 24 hours, title released in the future |
| `cross_table` | inconsistent with another table | principal who died before the title |
| `inconsistent_value` | fields of a row that contradict each other | season number without episode number |
| `orphan_foreign_key` | id pointing to a row that does not exist | `nconst` missing from `name.basics` |
| `duplicate` | two rows with the same key | two rows for the same `tconst` |

Thresholds (in `src/imdb_project/quality.py`):

- maximum plausible age: `MAX_PLAUSIBLE_AGE` years;
- first film year: `FIRST_FILM_YEAR` (no title can start before);
- "current year": the year the job runs; later start years are in the
  future.

In [14]:
print("MAX_PLAUSIBLE_AGE =", MAX_PLAUSIBLE_AGE, "| FIRST_FILM_YEAR =", FIRST_FILM_YEAR)
issues = report("quality_issues")
issues[issues["affected_rows"] > 0].sort_values(["category", "affected_rows"], ascending=[True, False])

MAX_PLAUSIBLE_AGE = 110 | FIRST_FILM_YEAR = 1874


,table,category,check,description,affected_rows,total_rows,affected_pct
19,title.principals,cross_table,principal_died_before_title,principal died before the title's startYear (all categories),384574,102641168,0.3747
20,title.principals,cross_table,principal_died_before_title_not_archive,"same, excluding archive_footage / archive_sound",287308,102641168,0.2799
21,title.episode,cross_table,episode_before_series,episode startYear is earlier than its series startYear,3536,10025083,0.0353
18,title.principals,cross_table,principal_born_after_title,principal born after the title's startYear,3101,102641168,0.0030
7,title.basics,implausible_value,start_in_future,startYear is after 2026 (announced or wrong),6017,12951113,0.0465
4,name.basics,implausible_value,probably_missing_death_year,no deathYear but born more than 110 years ago,4314,15802420,0.0273
9,title.basics,implausible_value,runtime_over_24h,runtimeMinutes is over 1440 (24 hours),209,12951113,0.0016
3,name.basics,implausible_value,lifespan_over_max,deathYear - birthYear is over 110 years,39,15802420,0.0002
0,name.basics,impossible_value,death_before_birth,deathYear is earlier than birthYear,28,15802420,0.0002
8,title.basics,impossible_value,non_positive_runtime,runtimeMinutes is 0 or negative,4,12951113,0.0000


Checks that found nothing are also part of the result: no duplicate key, no
rating outside [1, 10], no `endYear` on a title that is not a series...

In [15]:
issues[issues["affected_rows"] == 0][["table", "check", "description"]]

,table,check,description
1,name.basics,birth_in_future,birthYear is after 2026
2,name.basics,death_in_future,deathYear is after 2026
5,title.basics,end_before_start,endYear is earlier than startYear
6,title.basics,start_before_cinema,startYear is before 1874
10,title.basics,invalid_is_adult,isAdult is not 0 or 1
11,title.basics,end_year_on_non_series,endYear is set on a title that is not a series
12,title.akas,invalid_is_original_title,isOriginalTitle is not 0 or 1
13,title.episode,episode_is_own_parent,parentTconst equals tconst
14,title.episode,negative_season_or_episode,seasonNumber or episodeNumber is negative
15,title.episode,episode_number_without_season,only one of seasonNumber / episodeNumber is set


### 3.1 Looking at the issues

A count alone does not say whether the rows are errors. The examples below
look at the largest issues.

#### People who died before they were born (28)

Two different cases:

- **ancient authors** (Homer, Plato, Sophocles...): their years are BC but
  are stored without a minus sign, so the death year is smaller than the
  birth year;
- **modern people with a typing error** in one of the two years; which one
  is wrong cannot be known from the data.

In [16]:
(bronze["name.basics"]
 .where(F.col("deathYear").cast("int") < F.col("birthYear").cast("int"))
 .select("nconst", "primaryName", "birthYear", "deathYear", "primaryProfession")
 .orderBy(F.col("birthYear").cast("int")).toPandas())

,nconst,primaryName,birthYear,deathYear,primaryProfession
0,nm0653992,Ovid,43,17,"writer,music_department"
1,nm0515385,Titus Livius,59,17,writer
2,nm0549460,Publius Vergilius Maro,70,19,writer
3,nm2471712,Gaio Giulio Cesare,100,44,"writer,archive_footage"
4,nm0686974,Titus Maccius Plautus,254,184,writer
5,nm12436700,Aristotle,384,322,None
6,nm16507154,Lysippos,390,300,art_department
7,nm0686817,Plato,427,347,writer
8,nm1414200,Xenophon,430,354,writer
9,nm0034922,Aristophanes,450,388,"writer,soundtrack"


#### Principals who died before the title (287,308, archive footage excluded)

This is the largest issue, and most rows are **true**: the authors of
adapted works are credited as writers long after their death (J.R.R. Tolkien
for *The Lord of the Rings*, Bob Kane for *The Dark Knight*), and some actors
died during or after the shooting. Archive footage is already excluded from
the count.

In [17]:
films = (silver["title.basics"].where("titleType = 'movie'")
         .join(silver["title.ratings"], "tconst")
         .select("tconst", "primaryTitle", "startYear", "numVotes"))
(silver["title.principals"]
 .where(F.array_contains("dq_flags", "principal_died_before_title"))
 .join(films, "tconst")
 .join(silver["name.basics"].select("nconst", "primaryName", "deathYear"), "nconst")
 .orderBy(F.desc("numVotes"))
 .select("primaryName", "deathYear", "category", "primaryTitle", "startYear", "numVotes")
 .limit(10).toPandas())

,primaryName,deathYear,category,primaryTitle,startYear,numVotes
0,Bob Kane,1998,writer,The Dark Knight,2008,3249082
1,J.R.R. Tolkien,1973,writer,The Lord of the Rings: The Fellowship of the Ring,2001,2251341
2,J.R.R. Tolkien,1973,writer,The Lord of the Rings: The Return of the King,2003,2207007
3,Bob Kane,1998,writer,The Dark Knight Rises,2012,2037837
4,J.R.R. Tolkien,1973,writer,The Lord of the Rings: The Two Towers,2002,1996185
5,Oliver Reed,1999,actor,Gladiator,2000,1894108
6,Bob Kane,1998,writer,Batman Begins,2005,1753706
7,Bob Kane,1998,writer,Joker,2019,1736301
8,Bill Finger,1974,writer,Joker,2019,1736301
9,Leigh Brackett,1978,writer,Star Wars: Episode V - The Empire Strikes Back,1980,1532729


#### Principals born after the title (3,099)

Mostly **music written later for a restored version** of a silent film, or
**new dubbing** of an old film (*Fantasia*, 1940, was dubbed again in 1990).

In [18]:
(silver["title.principals"]
 .where(F.array_contains("dq_flags", "principal_born_after_title"))
 .join(films, "tconst")
 .join(silver["name.basics"].select("nconst", "primaryName", "birthYear"), "nconst")
 .orderBy(F.desc("numVotes"))
 .select("primaryName", "birthYear", "category", "primaryTitle", "startYear", "numVotes")
 .limit(10).toPandas())

,primaryName,birthYear,category,primaryTitle,startYear,numVotes
0,Robert Steadman,1965,composer,Nosferatu,1922,125275
1,Tim Matheson,1947,actor,Fantasia,1940,112841
2,Tim Matheson,1947,actor,Fantasia,1940,112841
3,Corey Burton,1955,actor,Fantasia,1940,112841
4,Hugh Douglas,1944,actor,Fantasia,1940,112841
5,Hugh Douglas,1944,actor,Fantasia,1940,112841
6,Corey Burton,1955,actor,Fantasia,1940,112841
7,Carl Davis,1936,composer,The General,1926,106733
8,Robert Israel,1963,composer,The General,1926,106733
9,Robert Israel,1963,composer,Sherlock Jr.,1924,67555


#### Titles with suspicious years or runtimes

`start_in_future` titles are mostly announced films and series. The table
shows the most voted titles with at least one flag. Runtimes over 24 hours
are of two kinds: series, audio books or games whose runtime is the total of
all their parts (*The Bear*, 1,611 minutes), and real experimental films
made to be very long (*The Cure for Insomnia*, 87 hours). Both are kept and
flagged.

In [19]:
(silver["title.basics"].where(F.size("dq_flags") > 0)
 .select("tconst", "primaryTitle", "titleType", "startYear", "runtimeMinutes", "dq_flags")
 .join(silver["title.ratings"].select("tconst", "numVotes"), "tconst", "left")
 .orderBy(F.desc_nulls_last("numVotes")).limit(10).toPandas())

,tconst,primaryTitle,titleType,startYear,runtimeMinutes,dq_flags,numVotes
0,tt14452776,The Bear,tvSeries,2022,1611,[runtime_over_24h],330075
1,tt13380510,A Game of Thrones (A Song of Ice and Fire - Book 1),video,2003,2026,[runtime_over_24h],24910
2,tt4507428,NBA 2K16,videoGame,2015,1495,[runtime_over_24h],925
3,tt13380758,Fire & Blood,video,2018,1584,[runtime_over_24h],527
4,tt0284020,The Cure for Insomnia,movie,1987,5220,[runtime_over_24h],448
5,tt8273150,Logistics,movie,2012,51420,[runtime_over_24h],414
6,tt0804414,Desire,tvSeries,2006,3900,[runtime_over_24h],345
7,tt0764146,The Other Day 1961-2003: Our Era,tvSeries,1997,1932,[runtime_over_24h],340
8,tt1230096,World War 2: The Complete History,tvMiniSeries,2000,1682,[runtime_over_24h],262
9,tt32397736,Harry Potter and the Order of the Phoenix: Full-Cast Edition - Book 5,video,2026,1599,[runtime_over_24h],255


## 4. Handling the issues

### 4.1 The rules

`jobs/prepare_data.py` builds the **silver** tables from bronze, with the
functions of `src/imdb_project/preparation.py`. Each issue is handled
according to how sure we are that the value is wrong:

| Level | Rule | Applied to |
|---|---|---|
| The value is **impossible** | set to null | both years of a person who died before being born; runtime of 0 minutes or less; `endYear` before `startYear` |
| The value is **possible but suspicious** | kept, and the name of the check is added to the `dq_flags` column | principal born after / died before the title, episode older than its series, runtime over 24 h, title in the future, lifespan over the maximum, probably missing death year, unknown person in `title.principals` |
| The row **cannot be linked** to a title | row dropped | rows of `title.akas`, `title.crew`, `title.ratings`, `title.episode`, `title.principals` whose title is missing from `title.basics` |
| An id in a **list** points to nobody | id removed from the list, row kept and flagged | unknown people in the `directors` / `writers` lists of `title.crew` |

Besides, every column gets its real type (integers, decimals), `0` / `1`
columns become booleans, comma-separated columns become arrays and the
`characters` column (JSON in the source) is parsed.

### 4.2 Why flag instead of delete?

Part 3 showed that most suspicious rows are true (adapted authors, restored
films). Deleting them would remove, for instance, J.R.R. Tolkien from *The
Lord of the Rings*. The flagged rows are also concentrated on famous films:

In [20]:
rated_films = films.where("numVotes >= 100")
flagged = (silver["title.principals"].where(F.size("dq_flags") > 0)
           .join(rated_films.select("tconst"), "tconst", "left_semi"))
print(f"Feature films with at least 100 votes: {rated_films.count():,}")
print(f"... with at least one flagged credit: {flagged.select('tconst').distinct().count():,}")

Feature films with at least 100 votes: 148,067


... with at least one flagged credit: 10,150


Each later analysis can still exclude flagged rows with one filter, see
part 5.

### 4.3 Result

Rows in / out, rows dropped and rows flagged, per table:

In [21]:
report("cleaning_summary")

,table,rows_in,rows_out,dropped_rows,flagged_rows
0,title.basics,12951113,12951113,0,6230
1,name.basics,15802420,15802420,0,4381
2,title.akas,60321594,60321592,2,0
3,title.crew,12949786,12949756,30,989
4,title.ratings,1730885,1730885,0,0
5,title.episode,10025088,10025083,5,3536
6,title.principals,102649609,102649609,0,298848


In [22]:
rows = []
for table in IMDB_TABLES:
    counts = (silver[table].select(F.explode("dq_flags").alias("flag"))
              .groupBy("flag").count().collect())
    rows += [(table, r["flag"], r["count"]) for r in counts]
pd.DataFrame(rows, columns=["table", "flag", "rows"]).sort_values(["table", "rows"], ascending=[True, False])

,table,flag,rows
0,name.basics,probably_missing_death_year,4314
1,name.basics,lifespan_over_max,39
2,name.basics,death_before_birth,28
5,title.basics,start_in_future,6017
4,title.basics,runtime_over_24h,209
3,title.basics,non_positive_runtime,4
7,title.crew,orphan_writer_removed,780
6,title.crew,orphan_director_removed,401
8,title.episode,episode_before_series,3536
9,title.principals,principal_died_before_title,287308


Example of the list cleaning in `title.crew`: the unknown ids were removed
from `writers`, the row is kept and flagged.

In [23]:
(silver["title.crew"].where(F.size("dq_flags") > 0)
 .join(bronze["title.crew"].select("tconst", F.col("writers").alias("writers_in_source")), "tconst")
 .select("tconst", "writers_in_source", "writers", "dq_flags").limit(5).toPandas())

,tconst,writers_in_source,writers,dq_flags
0,tt2400517,"nm18593213,nm18593219",[nm18593213],[orphan_writer_removed]
1,tt27832419,"nm17754681,nm16686595,nm17754680,nm2534689,nm1747813,nm14925016,nm14844348,n...","[nm16686595, nm17754680, nm2534689, nm1747813, nm14925016, nm14844348, nm516...",[orphan_writer_removed]
2,tt38623888,None,None,[orphan_director_removed]
3,tt38632385,None,None,[orphan_director_removed]
4,tt38641824,"nm17761442,nm17761441",None,[orphan_writer_removed]


## 5. Using the clean tables

The later steps read the silver tables:

```python
from imdb_project.io import read_silver
titles = read_silver(spark, "title.basics")
```

To keep only the rows without any flag:

```python
clean = titles.where(F.size("dq_flags") == 0)
```

or to exclude one issue only:

```python
principals.where(~F.array_contains("dq_flags", "principal_born_after_title"))
```

In [24]:
spark.stop()